In [7]:
import tkinter as tk
from tkinter import ttk, messagebox
import time

districts = [
    "Quận 1", "Quận 3", "Quận 4", "Quận 5", "Quận 10", "Phú Nhuận", "Bình Thạnh",
    "Quận 7", "Quận 8", "Quận 11", "Tân Bình", "Gò Vấp", "Quận 12", "TP. Thủ Đức"
]

neighbors = {
    "Quận 1": ["Quận 3", "Quận 4", "Quận 5", "Bình Thạnh", "Phú Nhuận", "TP. Thủ Đức"],
    "Quận 3": ["Quận 1", "Quận 10", "Phú Nhuận", "Quận 5", "Tân Bình"],
    "Quận 4": ["Quận 1", "Quận 7", "Quận 8"],
    "Quận 5": ["Quận 1", "Quận 3", "Quận 4", "Quận 10", "Quận 8", "Quận 11"],
    "Quận 10": ["Quận 3", "Quận 5", "Quận 11", "Tân Bình"],
    "Phú Nhuận": ["Quận 1", "Quận 3", "Bình Thạnh", "Tân Bình", "Gò Vấp"],
    "Bình Thạnh": ["Quận 1", "Phú Nhuận", "Gò Vấp", "TP. Thủ Đức"],
    "Quận 7": ["Quận 4", "Quận 8", "TP. Thủ Đức"],
    "Quận 8": ["Quận 4", "Quận 5", "Quận 7", "Quận 11"],
    "Quận 11": ["Quận 5", "Quận 10", "Quận 8", "Tân Bình"],
    "Tân Bình": ["Quận 3", "Quận 10", "Phú Nhuận", "Quận 11", "Gò Vấp", "Quận 12"],
    "Gò Vấp": ["Phú Nhuận", "Bình Thạnh", "Tân Bình", "Quận 12"],
    "Quận 12": ["Tân Bình", "Gò Vấp", "TP. Thủ Đức"],
    "TP. Thủ Đức": ["Quận 1", "Bình Thạnh", "Quận 7", "Quận 12"]
}

positions = {
    "Quận 12":     (180, 80),
    "Gò Vấp":      (340, 110),
    "TP. Thủ Đức": (560, 160),
    "Tân Bình":    (150, 220),
    "Phú Nhuận":   (320, 230),
    "Bình Thạnh":  (460, 250),
    "Quận 11":     (90,  340),
    "Quận 10":     (210, 330),
    "Quận 3":      (320, 340),
    "Quận 1":      (420, 370),
    "Quận 5":      (190, 440),
    "Quận 4":      (390, 470),
    "Quận 8":      (150, 540),
    "Quận 7":      (460, 550)
}

color_map = {
    "Đỏ": "#FF4D4D",
    "Xanh lá": "#2ECC71",
    "Xanh dương": "#3498DB",
    "Vàng": "#F1C40F",
    "Cam": "#E67E22"
}

class CSPVisualizer:
    def __init__(self, root):
        self.root = root
        self.root.title("Pure Backtracking Graph Coloring Visualizer")
        self.root.geometry("1000x650")
        self.root.configure(bg="#F5F6FA")

        self.is_running = False
        self.is_paused = False
        self.step_counter = 1
        self.assignment = {}
        self.available_colors = []

        self.create_widgets()
        self.draw_map_base()

    def create_widgets(self):
        left_frame = tk.Frame(self.root, width=350, bg="#FFFFFF", bd=2, relief=tk.GROOVE)
        left_frame.pack(side=tk.LEFT, fill=tk.Y, padx=10, pady=10)
        left_frame.pack_propagate(False)

        lbl_title = tk.Label(left_frame, text="BẢNG ĐIỀU KHIỂN", font=("Helvetica", 14, "bold"), bg="#FFFFFF", fg="#2F3640")
        lbl_title.pack(pady=10)

        lbl_color = tk.Label(left_frame, text="Chọn số lượng màu sử dụng:", font=("Helvetica", 10), bg="#FFFFFF")
        lbl_color.pack(anchor="w", padx=15, pady=2)

        self.cbo_colors = ttk.Combobox(left_frame, values=["3 Màu (Đỏ, Xanh lá, Xanh dương)", "4 Màu (Đỏ, Xanh lá, Xanh dương, Vàng)", "5 Màu (Đỏ, Xanh lá, Xanh dương, Vàng, Cam)"], state="readonly")
        self.cbo_colors.current(0)
        self.cbo_colors.pack(fill=tk.X, padx=15, pady=5)

        btn_frame = tk.Frame(left_frame, bg="#FFFFFF")
        btn_frame.pack(pady=15, fill=tk.X, padx=15)

        self.btn_start = tk.Button(btn_frame, text="Bắt đầu", font=("Helvetica", 10, "bold"), bg="#4CD137", fg="white", width=10, command=self.start_algorithm)
        self.btn_start.grid(row=0, column=0, pady=5, padx=2)

        self.btn_pause = tk.Button(btn_frame, text="Tạm dừng", font=("Helvetica", 10, "bold"), bg="#FBC531", fg="white", width=10, state=tk.DISABLED, command=self.toggle_pause)
        self.btn_pause.grid(row=0, column=1, pady=5, padx=2)

        self.btn_reset = tk.Button(btn_frame, text="Làm mới", font=("Helvetica", 10, "bold"), bg="#EA2027", fg="white", width=10, command=self.reset_visualizer)
        self.btn_reset.grid(row=0, column=2, pady=5, padx=2)

        lbl_log = tk.Label(left_frame, text="Output:", font=("Helvetica", 10, "bold"), bg="#FFFFFF", fg="#2F3640")
        lbl_log.pack(anchor="w", padx=15, pady=(10, 2))

        self.txt_log = tk.Text(left_frame, bg="#2F3640", fg="#FFFFFF", font=("Courier New", 9), wrap=tk.WORD)
        self.txt_log.pack(fill=tk.BOTH, expand=True, padx=15, pady=10)

        right_frame = tk.Frame(self.root, bg="#F5F6FA")
        right_frame.pack(side=tk.RIGHT, fill=tk.BOTH, expand=True, padx=10, pady=10)

        lbl_map_title = tk.Label(right_frame, text="BẢN ĐỒ SƠ ĐỒ CÁC QUẬN TP.HCM", font=("Helvetica", 14, "bold"), bg="#F5F6FA", fg="#2F3640")
        lbl_map_title.pack(pady=5)

        self.canvas = tk.Canvas(right_frame, bg="#FFFFFF", bd=2, relief=tk.SUNKEN, highlightbackground="#DCDDE1")
        self.canvas.pack(fill=tk.BOTH, expand=True)

        self.node_objects = {}

    def draw_map_base(self):
        self.canvas.delete("all")
        self.node_objects.clear()

        drawn_edges = set()
        for node, adjacent in neighbors.items():
            for adj in adjacent:
                edge = tuple(sorted((node, adj)))
                if edge not in drawn_edges:
                    if node in positions and adj in positions:
                        x1, y1 = positions[node]
                        x2, y2 = positions[adj]
                        self.canvas.create_line(x1, y1, x2, y2, width=2, fill="#B2BEC3", tags="edge")
                        drawn_edges.add(edge)

        radius = 32
        for node, (x, y) in positions.items():
            circle_id = self.canvas.create_oval(x - radius, y - radius, x + radius, y + radius,
                                                fill="#FFFFFF", outline="#636E72", width=2)
            self.node_objects[node] = circle_id
            self.canvas.create_text(x, y, text=node, font=("Helvetica", 9, "bold"), fill="#2D3436")

    def update_node_color(self, district_name, color_name):
        color_hex = color_map.get(color_name, "#FFFFFF") if color_name else "#FFFFFF"
        if district_name in self.node_objects:
            self.canvas.itemconfig(self.node_objects[district_name], fill=color_hex)
        self.root.update()

    def log(self, text):
        self.txt_log.insert(tk.END, text + "\n")
        self.txt_log.see(tk.END)
        self.root.update()

    def get_selected_colors(self):
        idx = self.cbo_colors.current()
        all_avail = ["Đỏ", "Xanh lá", "Xanh dương", "Vàng", "Cam"]
        return all_avail[:3 + idx]

    def check_pause(self):
        while self.is_paused:
            self.root.update()
            time.sleep(0.1)

    def toggle_pause(self):
        if self.is_paused:
            self.is_paused = False
            self.btn_pause.config(text="Tạm dừng", bg="#FBC531")
            self.log("Tiếp tục chạy...")
        else:
            self.is_paused = True
            self.btn_pause.config(text="Tiếp tục", bg="#4CD137")
            self.log("Đã tạm dừng.")

    def start_algorithm(self):
        if self.is_running:
            return

        self.is_running = True
        self.is_paused = False
        self.btn_start.config(state=tk.DISABLED)
        self.btn_pause.config(state=tk.NORMAL)
        self.cbo_colors.config(state=tk.DISABLED)

        self.available_colors = self.get_selected_colors()
        self.assignment = {}
        self.step_counter = 1
        self.txt_log.delete("1.0", tk.END)

        self.log("=== BACKTRACKING ===")
        self.log(f"Màu sử dụng: {', '.join(self.available_colors)}")

        success = self.solve_csp_pure_backtracking()

        if success:
            self.log("\n=============================\nKẾT QUẢ: Thành công!")
            messagebox.showinfo("Thành công", "Đã tìm thấy phương án tô màu hợp lệ!")
        else:
            self.log("\n=============================\nKẾT QUẢ: Thất bại! Không tìm thấy lời giải.")
            messagebox.showerror("Thất bại", "Không tìm thấy lời giải với số màu này!")

        self.btn_pause.config(state=tk.DISABLED)
        self.is_running = False

    def reset_visualizer(self):
        self.is_running = False
        self.is_paused = False
        self.btn_start.config(state=tk.NORMAL)
        self.btn_pause.config(text="Tạm dừng", state=tk.DISABLED)
        self.cbo_colors.config(state="readonly")
        self.txt_log.delete("1.0", tk.END)
        self.draw_map_base()
        self.log("Đã làm mới trạng thái.")

    def is_valid(self, var, color):
        for neighbor in neighbors[var]:
            if neighbor in self.assignment:
                if self.assignment[neighbor] == color:
                    return False
        return True

    def solve_csp_pure_backtracking(self):
        if len(self.assignment) == len(districts):
            return True

        var = [d for d in districts if d not in self.assignment][0]

        self.log(f"\n--- Bước {self.step_counter}: Chọn {var} ---")
        self.step_counter += 1

        for color in self.available_colors:
            self.check_pause()
            if not self.is_running: return False

            self.log(f" * Thử gán {var} = {color}")

            if self.is_valid(var, color):
                self.assignment[var] = color
                self.update_node_color(var, color)
                self.log(f"   => Hợp lệ! Tiếp tục chọn quận tiếp theo.")
                time.sleep(0.6)

                if self.solve_csp_pure_backtracking():
                    return True

            else:
                self.log(f"   => Xung đột! Trùng màu với hàng xóm.")
                self.update_node_color(var, color)
                time.sleep(0.6)
                self.update_node_color(var, None)

            self.check_pause()
            if var in self.assignment:
                self.log(f" -> [Backtracking] Bỏ màu của {var}")
                del self.assignment[var]
                self.update_node_color(var, None)
                time.sleep(0.6)

        return False

if __name__ == "__main__":
    root = tk.Tk()
    app = CSPVisualizer(root)
    root.mainloop()

Exception in Tkinter callback
Traceback (most recent call last):
  File "C:\Users\ASUS\AppData\Local\Programs\Python\Python313\Lib\tkinter\__init__.py", line 2068, in __call__
    return self.func(*args)
           ~~~~~~~~~^^^^^^^
  File "C:\Users\ASUS\AppData\Local\Temp\ipykernel_9064\1153918255.py", line 183, in start_algorithm
    success = self.solve_csp_pure_backtracking()
  File "C:\Users\ASUS\AppData\Local\Temp\ipykernel_9064\1153918255.py", line 233, in solve_csp_pure_backtracking
    if self.solve_csp_pure_backtracking():
       ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^^
  File "C:\Users\ASUS\AppData\Local\Temp\ipykernel_9064\1153918255.py", line 233, in solve_csp_pure_backtracking
    if self.solve_csp_pure_backtracking():
       ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^^
  File "C:\Users\ASUS\AppData\Local\Temp\ipykernel_9064\1153918255.py", line 233, in solve_csp_pure_backtracking
    if self.solve_csp_pure_backtracking():
       ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^^
  [Previous line repeate